In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats

# Expression Correlation


## Data Source: 
https://www.gtexportal.org/home/downloads/adult-gtex/bulk_tissue_expression (TPM) 

# BIOMARKER VS QUERY

In [ ]:
main_file_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv')
biomarker_query = main_file_csv[['ensembl_gene_id_biomarker','ensembl_gene_id_query']]
biomarker_query = biomarker_query.rename(columns={
    'ensembl_gene_id_biomarker': 'A1_ensembl',
    'ensembl_gene_id_query' : 'A2_ensembl'
})
biomarker_query_genes = pd.unique(biomarker_query[['A1_ensembl', 'A2_ensembl']].values.ravel())
biomarker_query_genes

In [ ]:
biomarker_query

In [ ]:
df_biomarker_query_genes = pd.DataFrame()

for chunk in pd.read_csv('input_data/GTEx/GTEx_Analysis_v10_RNASeQCv2.4.2_gene_tpm.gct.gz', sep='\t', skiprows=2, iterator=True, chunksize=1000):
    chunk_df = chunk.assign(ensembl_id = chunk.Name.apply(lambda x: x.split('.')[0])).drop(columns=['Description'])
    df_biomarker_query_genes = pd.concat([df_biomarker_query_genes, chunk_df[chunk_df.ensembl_id.isin(biomarker_query_genes)].set_index('ensembl_id')])



In [ ]:
df_biomarker_query_genes

In [ ]:
# Now create funtions 
def CalculateMeanExpression(expr_data):
    mean_expr = pd.DataFrame(expr_data.mean(axis=1), columns=['mean_expr']).reset_index()

    return mean_expr

def ExpandToPairs(mean_expr, pairs):
    pairs_expr = pd.merge(pairs, mean_expr.rename(columns={'ensembl_id':'A1_ensembl', 'mean_expr':'A1_mean_expr'}), how='left')
    
    pairs_expr = pd.merge(pairs_expr, mean_expr.rename(columns={'ensembl_id':'A2_ensembl', 'mean_expr':'A2_mean_expr'}), how='left')
    
    print('Pairs where both genes have expr:', 
    pairs_expr[(~pairs_expr.A1_mean_expr.isna()) & (~pairs_expr.A2_mean_expr.isna())].shape[0], '/', pairs_expr.shape[0])
    pairs_expr[:1]
    # Filter pairs down to those where both have expression (not NA and not 0)
    pairs_expr_subset = pairs_expr[(~pairs_expr.A1_mean_expr.isna()) & (~pairs_expr.A2_mean_expr.isna()) & 
                               (pairs_expr.A1_mean_expr!=0) & (pairs_expr.A2_mean_expr!=0)]
    print('# Pairs where both have expression:', pairs_expr_subset.shape[0])
    print('# Total Pairs:', pairs.shape[0])

    return pairs_expr
    
def CalculateExprCorrelation(pairs, expr_data):
    
    #display(expr_data)
    mean_expr = CalculateMeanExpression(expr_data)
    #display(mean_expr)
    pairs_expr_subset = ExpandToPairs(mean_expr, pairs)
    display(pairs_expr_subset)
    def compute_spearman_corr(pair):

        return stats.spearmanr(
            expr_data.loc[pair.A1_ensembl],
            expr_data.loc[pair.A2_ensembl])[0]

    #display(pairs_expr_subset)
    features_spearman = pairs_expr_subset.assign(spearman_corr = pairs_expr_subset.apply(compute_spearman_corr, axis=1))
    display(features_spearman[:1])
    return(features_spearman)


def CalculateExprCorrelation_Fast(pairs, expr_data):
    
    #display(expr_data)
    mean_expr = CalculateMeanExpression(expr_data)
    print(mean_expr.shape[0])

    # Expand pairs
    pairs_expr_subset = ExpandToPairs(mean_expr, pairs)
    print(pairs_expr_subset.shape[0])


    # 🔹 **Optimized Spearman Correlation Calculation**
    # Convert expressions to ranks (Spearman is Pearson on ranked data)
    expr_ranks = expr_data.rank(axis=1)  # Rank across rows

    # Prepare dictionary for faster lookup
    expr_dict = expr_ranks.to_dict(orient="index")

    # Compute correlation using NumPy
    def fast_spearman(pair):
        
        A1 = expr_dict.get(pair.A1_ensembl, None)
        A2 = expr_dict.get(pair.A2_ensembl, None)
        #print(A1)
        #print(A2)
        if A1 is None or A2 is None:
            return np.nan  # Missing data
        return np.corrcoef(list(A1.values()), list(A2.values()))[0, 1]

    # Apply optimized function
    pairs_expr_subset["spearman_corr"] = pairs_expr_subset.apply(fast_spearman, axis=1)
    print(pairs_expr_subset.shape[0])
    return pairs_expr_subset


In [ ]:
# 🔹 Calculate Mean Expression
def CalculateMeanExpression(expr_data):
    mean_expr = pd.DataFrame(expr_data.mean(axis=1), columns=['mean_expr']).reset_index()
    return mean_expr

# 🔹 Calculate Expression Variance
def CalculateVarianceExpression(expr_data):
    var_expr = pd.DataFrame(expr_data.var(axis=1), columns=['expr_variance']).reset_index()
    return var_expr

# 🔹 Expand gene pairs with mean and variance expression
def ExpandToPairs(mean_expr, var_expr, pairs):
    # Add mean expression
    pairs_expr = pd.merge(pairs, mean_expr.rename(columns={'ensembl_id':'A1_ensembl', 'mean_expr':'A1_mean_expr'}), how='left')
    pairs_expr = pd.merge(pairs_expr, mean_expr.rename(columns={'ensembl_id':'A2_ensembl', 'mean_expr':'A2_mean_expr'}), how='left')

    # Add variance expression
    pairs_expr = pd.merge(pairs_expr, var_expr.rename(columns={'ensembl_id':'A1_ensembl', 'expr_variance':'A1_expr_variance'}), how='left')
    pairs_expr = pd.merge(pairs_expr, var_expr.rename(columns={'ensembl_id':'A2_ensembl', 'expr_variance':'A2_expr_variance'}), how='left')

    print('Pairs where both genes have expr:', 
          pairs_expr[(~pairs_expr.A1_mean_expr.isna()) & (~pairs_expr.A2_mean_expr.isna())].shape[0], '/', pairs_expr.shape[0])
    
    # Filter pairs down to those where both have non-zero, non-NA expression
    pairs_expr_subset = pairs_expr[
        (~pairs_expr.A1_mean_expr.isna()) & (~pairs_expr.A2_mean_expr.isna()) & 
        (pairs_expr.A1_mean_expr != 0) & (pairs_expr.A2_mean_expr != 0)
    ]
    print('# Pairs where both have expression:', pairs_expr_subset.shape[0])
    print('# Total Pairs:', pairs.shape[0])

    return pairs_expr

# 🔹 Original Spearman Correlation Function
def CalculateExprCorrelation(pairs, expr_data):
    mean_expr = CalculateMeanExpression(expr_data)
    var_expr = CalculateVarianceExpression(expr_data)

    pairs_expr_subset = ExpandToPairs(mean_expr, var_expr, pairs)

    def compute_spearman_corr(pair):
        return stats.spearmanr(expr_data.loc[pair.A1_ensembl], expr_data.loc[pair.A2_ensembl])[0]

    features_spearman = pairs_expr_subset.assign(spearman_corr = pairs_expr_subset.apply(compute_spearman_corr, axis=1))
    return features_spearman

# 🔹 Fast Spearman Correlation Version
def CalculateExprCorrelation_Fast(pairs, expr_data):
    mean_expr = CalculateMeanExpression(expr_data)
    var_expr = CalculateVarianceExpression(expr_data)

    pairs_expr_subset = ExpandToPairs(mean_expr, var_expr, pairs)

    expr_ranks = expr_data.rank(axis=1)  # Rank across samples (columns)
    expr_dict = expr_ranks.to_dict(orient="index")

    def fast_spearman(pair):
        A1 = expr_dict.get(pair.A1_ensembl, None)
        A2 = expr_dict.get(pair.A2_ensembl, None)
        if A1 is None or A2 is None:
            return np.nan
        return np.corrcoef(list(A1.values()), list(A2.values()))[0, 1]

    pairs_expr_subset["spearman_corr"] = pairs_expr_subset.apply(fast_spearman, axis=1)
    return pairs_expr_subset


In [ ]:
duplicate_biomarker_query = df_biomarker_query_genes.index[df_biomarker_query_genes.index.duplicated()]
display(duplicate_biomarker_query)


In [ ]:
df_biomarker_query_genes = df_biomarker_query_genes.reset_index()
df_biomarker_query_genes

In [ ]:
duplicates = df_biomarker_query_genes[df_biomarker_query_genes.duplicated(subset=["ensembl_id"], keep=False)]
non_zero_rows = duplicates[(duplicates.iloc[:, 2:] != 0).any(axis=1)]
expr_data_biomarker_query = df_biomarker_query_genes.drop(non_zero_rows.index)

expr_data_biomarker_query = expr_data_biomarker_query.set_index("ensembl_id")
expr_data_biomarker_query.drop(columns=['Name'], inplace = True)
expr_data_biomarker_query

In [ ]:
biomarker_query_expression = CalculateExprCorrelation_Fast(biomarker_query,expr_data_biomarker_query)

In [ ]:
biomarker_query_expression.to_csv('input_data/3_ML_outputs/feature_output/gtex_co_expression_biomarker_query.csv',index=False)

# TARGET 1 vs Query

In [ ]:
main_file_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv')
target1_query = main_file_csv[['ensembl_gene_id_target1','ensembl_gene_id_query']]
target1_query = target1_query.rename(columns={
    'ensembl_gene_id_target1': 'A1_ensembl',
    'ensembl_gene_id_query' : 'A2_ensembl'
})
target1_query_genes = pd.unique(target1_query[['A1_ensembl', 'A2_ensembl']].values.ravel())
target1_query_genes

In [ ]:
target1_query

In [ ]:
df_target1_query_genes = pd.DataFrame()

for chunk in pd.read_csv('input_data/GTEx_Analysis_v10_RNASeQCv2.4.2_gene_tpm.gct.gz', sep='\t', skiprows=2, iterator=True, chunksize=1000):
    chunk_df = chunk.assign(ensembl_id = chunk.Name.apply(lambda x: x.split('.')[0])).drop(columns=['Description'])
    df_target1_query_genes = pd.concat([df_target1_query_genes, chunk_df[chunk_df.ensembl_id.isin(target1_query_genes)].set_index('ensembl_id')])



In [ ]:
df_target1_query_genes

In [ ]:
duplicate_target1_query = df_target1_query_genes.index[df_target1_query_genes.index.duplicated()]
display(duplicate_target1_query)


In [ ]:
df_target1_query_genes = df_target1_query_genes.reset_index()
df_target1_query_genes

In [ ]:
duplicates = df_target1_query_genes[df_target1_query_genes.duplicated(subset=["ensembl_id"], keep=False)]
non_zero_rows = duplicates[(duplicates.iloc[:, 2:] != 0).any(axis=1)]
expr_data_target1_query = df_target1_query_genes.drop(non_zero_rows.index)

expr_data_target1_query = expr_data_target1_query.set_index("ensembl_id")
expr_data_target1_query.drop(columns=['Name'], inplace = True)
expr_data_target1_query

In [ ]:
target1_query_expression = CalculateExprCorrelation_Fast(target1_query,expr_data_target1_query)

In [ ]:
target1_query_expression.to_csv('input_data/3_ML_outputs/feature_output/gtex_co_expression_target1_query.csv',index=False)

# Target 2 vs Query

In [ ]:
main_file_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv')
target2_query = main_file_csv[['ensembl_gene_id_target2','ensembl_gene_id_query']]
target2_query = target2_query.rename(columns={
    'ensembl_gene_id_target2': 'A1_ensembl',
    'ensembl_gene_id_query' : 'A2_ensembl'
})
target2_query_genes = pd.unique(target2_query[['A1_ensembl', 'A2_ensembl']].values.ravel())
target2_query_genes

In [ ]:
target2_query

In [ ]:
df_target2_query_genes = pd.DataFrame()

for chunk in pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/new_analysis_with_hit_selection/predicting_resistance/input_data/GTEx_Analysis_v10_RNASeQCv2.4.2_gene_tpm.gct.gz', sep='\t', skiprows=2, iterator=True, chunksize=1000):
    chunk_df = chunk.assign(ensembl_id = chunk.Name.apply(lambda x: x.split('.')[0])).drop(columns=['Description'])
    df_target2_query_genes = pd.concat([df_target2_query_genes, chunk_df[chunk_df.ensembl_id.isin(target2_query_genes)].set_index('ensembl_id')])



In [ ]:
df_target2_query_genes

In [ ]:
duplicate_target2_query = df_target2_query_genes.index[df_target2_query_genes.index.duplicated()]
display(duplicate_target2_query)


In [ ]:
df_target2_query_genes = df_target2_query_genes.reset_index()
df_target2_query_genes

In [ ]:
duplicates = df_target2_query_genes[df_target2_query_genes.duplicated(subset=["ensembl_id"], keep=False)]
non_zero_rows = duplicates[(duplicates.iloc[:, 2:] != 0).any(axis=1)]
expr_data_target2_query = df_target2_query_genes.drop(non_zero_rows.index)

expr_data_target2_query = expr_data_target2_query.set_index("ensembl_id")
expr_data_target2_query.drop(columns=['Name'], inplace = True)
expr_data_target2_query

In [ ]:
target2_query_expression = CalculateExprCorrelation_Fast(target2_query,expr_data_target2_query)

In [ ]:
target2_query_expression.to_csv('input_data/3_ML_outputs/feature_output/gtex_co_expression_target2_query.csv',index=False)